## Audit new drift-aware backtest engine

### Setup

In [1]:
import numpy as np
import pandas as pd

from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet
from alpha_research.portfolio import build_factor_target_weights


panel = load_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}

config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

### Run both engines

In [2]:
audit_results = {}

return_panel = panel[
    ["date", "ticker", "forward_ret_1d"]
].copy()

for factor_name, factor_column in factor_columns.items():
    target_weights = build_factor_target_weights(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    legacy_daily, legacy_holdings = run_long_short_backtest(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    drift_daily, drift_holdings = run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=target_weights,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )

    audit_results[factor_name] = {
        "targets": target_weights,
        "legacy_daily": legacy_daily,
        "legacy_holdings": legacy_holdings,
        "drift_daily": drift_daily,
        "drift_holdings": drift_holdings,
    }

### Verify the results

Both engines should match the target weights exactly on rebalance dates.

In [3]:
def maximum_weight_difference(
    left: pd.DataFrame,
    right: pd.DataFrame,
) -> float:
    comparison = left[["date", "ticker", "weight"]].merge(
        right[["date", "ticker", "weight"]],
        on=["date", "ticker"],
        how="outer",
        suffixes=("_left", "_right"),
    )

    comparison[["weight_left", "weight_right"]] = comparison[
        ["weight_left", "weight_right"]
    ].fillna(0.0)

    return float((comparison["weight_left"] - comparison["weight_right"]).abs().max())


audit_check_rows = []

for factor_name, result in audit_results.items():
    targets = result["targets"]
    target_dates = targets["date"].unique()

    legacy_rebalance_holdings = result["legacy_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    drift_rebalance_holdings = result["drift_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    daily_comparison = result["legacy_daily"][
        ["date", "is_rebalance", "gross_return"]
    ].merge(
        result["drift_daily"][["date", "is_rebalance", "gross_return"]],
        on="date",
        how="inner",
        suffixes=("_legacy", "_drift"),
        validate="one_to_one",
    )

    gross_difference = (
        daily_comparison["gross_return_drift"] - daily_comparison["gross_return_legacy"]
    ).abs()

    rebalance_mask = daily_comparison["is_rebalance_legacy"]

    audit_check_rows.append(
        {
            "factor": factor_name,
            "same_daily_dates": (
                len(daily_comparison)
                == len(result["legacy_daily"])
                == len(result["drift_daily"])
            ),
            "rebalance_dates": len(target_dates),
            "max_legacy_target_mismatch": (
                maximum_weight_difference(
                    legacy_rebalance_holdings,
                    targets,
                )
            ),
            "max_drift_target_mismatch": (
                maximum_weight_difference(
                    drift_rebalance_holdings,
                    targets,
                )
            ),
            "max_rebalance_gross_return_difference": (
                gross_difference.loc[rebalance_mask].max()
            ),
            "mean_non_rebalance_gross_return_difference": (
                gross_difference.loc[~rebalance_mask].mean()
            ),
        }
    )

audit_checks = pd.DataFrame(audit_check_rows).set_index("factor")

audit_checks

,same_daily_dates,rebalance_dates,max_legacy_target_mismatch,max_drift_target_mismatch,max_rebalance_gross_return_difference,mean_non_rebalance_gross_return_difference
factor,,,,,,
12-1 Momentum,True,578,0.0,0.0,0.0,0.000349
Realised Volatility,True,578,0.0,0.0,0.0,0.000262


#### Compare economic results

In [ ]:
summary_rows = []

for factor_name, result in audit_results.items():
    for engine_name, daily in {
        "Legacy": result["legacy_daily"],
        "Drift-aware": result["drift_daily"],
    }.items():
        gross = summarise_backtest(
            daily,
            return_column="gross_return",
        ).iloc[0]

        net = summarise_backtest(
            daily,
            return_column="net_return",
        ).iloc[0]

        summary_rows.append(
            {
                "factor": factor_name,
                "engine": engine_name,
                "gross_total_return": gross["total_return"],
                "net_total_return": net["total_return"],
                "gross_annualised_return": (gross["annualised_return"]),
                "net_annualised_return": (net["annualised_return"]),
                "net_annualised_volatility": (net["annualised_volatility"]),
                "gross_sharpe": gross["sharpe_ratio"],
                "net_sharpe": net["sharpe_ratio"],
                "average_rebalance_turnover": (net["average_rebalance_turnover"]),
                "total_transaction_cost": (net["total_transaction_cost"]),
            }
        )

audit_summary = (
    pd.DataFrame(summary_rows).sort_values(["factor", "engine"]).reset_index(drop=True)
)

audit_summary.round(4)

,factor,engine,gross_total_return,net_total_return,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,average_rebalance_turnover,total_transaction_cost
0,12-1 Momentum,Drift-aware,0.4804,0.1038,0.0348,0.0086,0.2119,0.2683,0.1475,0.5080,0.2936
1,12-1 Momentum,Legacy,0.5957,0.2379,0.0416,0.0188,0.2113,0.2995,0.1947,0.4393,0.2539
2,Realised Volatility,Drift-aware,5.5496,4.1786,0.1781,0.1542,0.2395,0.8042,0.7185,0.4063,0.2348
3,Realised Volatility,Legacy,5.5559,4.3711,0.1782,0.1579,0.2399,0.8036,0.7310,0.3448,0.1993


#### Isolate the change

In [ ]:
metric_columns = [
    "gross_annualised_return",
    "net_annualised_return",
    "net_annualised_volatility",
    "gross_sharpe",
    "net_sharpe",
    "average_rebalance_turnover",
    "total_transaction_cost",
]

legacy_summary = audit_summary.loc[audit_summary["engine"] == "Legacy"].set_index(
    "factor"
)

drift_summary = audit_summary.loc[audit_summary["engine"] == "Drift-aware"].set_index(
    "factor"
)

audit_deltas = drift_summary[metric_columns] - legacy_summary[metric_columns]

audit_deltas.columns = [f"change_in_{column}" for column in audit_deltas.columns]

audit_deltas.round(4)

,change_in_gross_annualised_return,change_in_net_annualised_return,change_in_net_annualised_volatility,change_in_gross_sharpe,change_in_net_sharpe,change_in_average_rebalance_turnover,change_in_total_transaction_cost
factor,,,,,,,
12-1 Momentum,-0.0068,-0.0101,0.0006,-0.0312,-0.0472,0.0687,0.0397
Realised Volatility,-0.0001,-0.0037,-0.0004,0.0006,-0.0125,0.0615,0.0355


### Engine audit conclusion

The drift-aware engine reproduces the legacy target portfolios exactly on rebalance dates. Rebalance-day gross returns also match exactly, confirming that the factor signals, stock selection, and target-weight construction are unchanged.

Between rebalances, the drift-aware engine carries forward holdings whose weights evolve with asset returns. This produces non-zero return differences relative to the legacy constant-weight convention and increases measured rebalance turnover.

The legacy implementation understated average rebalance turnover by approximately 0.069 for momentum and 0.062 for realised volatility. After realistic drift accounting, momentum's net Sharpe declines from approximately 0.195 to 0.148, while realised volatility's net Sharpe declines from 0.731 to 0.719.

All subsequent portfolio experiments will therefore use the drift-aware target-weight engine. Legacy results will be retained only as historical benchmarks.